In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

# Sample text
text = "hello world"
chars = sorted(list(set(text)))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# Prepare data
seq_length = 3
X, y = [], []
for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    target = text[i+seq_length]
    X.append([char_to_idx[c] for c in seq])
    y.append(char_to_idx[target])

X = np.array(X)
y = np.array(y)

# Build RNN model
model = Sequential([
    Embedding(input_dim=len(chars), output_dim=10, input_length=seq_length),
    SimpleRNN(20),
    Dense(len(chars), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=0)

# Predict next character
test = "wor"
x_pred = np.array([[char_to_idx[c] for c in test]])
pred_idx = np.argmax(model.predict(x_pred))
print(f"Input: {test} → Predicted next char: {idx_to_char[pred_idx]}")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Input: wor → Predicted next char: l


# Sample text
text = "hello world"


set(text) gets unique characters; sorted gives a stable order.

For "hello world" the unique chars are [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w'] (8 chars).

char_to_idx maps each character → integer index (0..7).

idx_to_char is the inverse mapping to convert predicted indices back to characters.

seq_length = 3
X, y = [], []
for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    target = text[i+seq_length]
    X.append([char_to_idx[c] for c in seq])
    y.append(char_to_idx[target])

X = np.array(X)
y = np.array(y)

seq_length = 3 → model looks at 3 previous chars to predict the next one.

Loop builds all sliding windows:

Example sliding windows:

i=0: seq = "hel", target = "l"

i=1: seq = "ell", target = "o"

...

i=7: seq = "wor", target = "l"

Number of training examples = len(text) - seq_length = 11 - 3 = 8.

After this:

X.shape == (8, 3) — 8 samples, each is 3 integer indexes.

y.shape == (8,) — 8 integer labels (each 0..7).

Example first few X,y (as indices):

X[0] = indices for "hel", y[0] = index for 'l', etc.

model = Sequential([
    Embedding(input_dim=len(chars), output_dim=10, input_length=seq_length),
    SimpleRNN(20),
    Dense(len(chars), activation='softmax')
])


Layer by layer:

Embedding

input_dim=len(chars) = 8 (vocabulary size).

output_dim=10 = embedding vector size for each char.

input_length=3 = sequence length.

Input to embedding: (batch_size, 3) of integers.

Output from embedding: (batch_size, 3, 10) (each char replaced by a 10-d vector).

SimpleRNN(20)

A standard RNN cell with 20 hidden units.

It reads the 3-step sequence of embeddings and returns the final hidden state (because return_sequences=False by default).

Output shape: (batch_size, 20).

Dense(len(chars), activation='softmax')

A fully connected layer that maps 20 → 8 logits, then softmax to get probabilities over the 8 possible next characters.

Output shape: (batch_size, 8) (probabilities for each character).

test = "wor"
x_pred = np.array([[char_to_idx[c] for c in test]])
pred_idx = np.argmax(model.predict(x_pred))
print(f"Input: {test} → Predicted next char: {idx_to_char[pred_idx]}")


Step-by-step:

test = "wor" is a 3-char input.

Convert to indices and wrap in an extra dimension → x_pred.shape == (1, 3) (a single batch).

model.predict(x_pred) returns a (1, 8) array of probabilities for the next char.

np.argmax(...) selects the index with highest probability.

idx_to_char[pred_idx] converts index back to a character and prints it.